In [1]:
from sagemaker.experiments import Run
from sagemaker.pytorch import PyTorch
from sagemaker import Session
from sagemaker import get_execution_role
from sagemaker.utils import unique_name_from_base
from sagemaker.debugger import TensorBoardOutputConfig
import os

# Notes on running on AWS SM:
# instance type: numGPUs, memPerGPU, max images per batch (this is a per GPU number), approximate time per epoch, cost, cost multiplier rel g4dn..
# "ml.g4dn.4xlarge": 1, 16, 20, 3.32 min per epoch, $1.806 per hr., x1
# "ml.p3.8xlarge": 4, 16, 20, 1.41 min per epoch, $14.688, x8.132
# "ml.p3.16xlarge": 8, 16, 20, 1min per epoch, $28.152, x15.588
# "ml.p4de.24xlarge": 8, 80, 20

# If running notebook inside a SageMaker Notebook or Studio instance,
# can retrieve the execution role associated with the instance
role = "arn:aws:iam::159929462505:role/LLNL_User_Roles_Sagemaker" #get_execution_role()

project_name = "sagemaker-test_seg_mz2"
run_name = "ml.g4dn.4xlarge"   
experiment_name = "segmentation"

train_image = "s3://image-segmentation-2k/images" #"s3://wwps-llnl-model-dev/Dataset-2K/images"
train_mask = "s3://image-segmentation-2k/masks" #"s3://wwps-llnl-model-dev/Dataset-2K/masks"

hyperparameters = {
    "num_workers": 8,
    "model_name": "UNet",
    "model_params": '''{"param": "value"}''',
    # Note, with Pytorch Lightning, this is a per GPU batch count. PL will create a batch per device
    # behind the scenes. No need to adjust if changing instances with different gpu counts. 
    # Recommend adjust if changes instances that have different GPU memory (e.g., p3.* -> p3dn.* -> p4d.* -> p4de.*)
    "batch_size": 20,
    "epochs": 100,
    "learning_rate": 1e-4,
    "lr_schedular": "cosine_warmup",
    "image_mode": "grayscale",
    # Added a new helper function called str2bool in train.py to aid in this problem.
    # Can now use "False", "No" or 0 to indicate a negative boolean
    # and "True", "Yes" or 1 to indicate a positive boolean.
    "use_zero":"True", # Note comment on bool as type. Empty String => "" = False; Any string => "Hello" | "True" | "Flase" = True; https://docs.python.org/3/library/argparse.html#type
    "use_amp": "True", # See above comment
    # Data Augmentation Parameters
    "image_size": 512,
    "use_random_resize": "True",
    "use_random_rotation": "False",
    "color_jitter": 0.0,
    "center_crop": "True",
    "center_crop_size": 1200,
    "test_image_size": 1200,
    "test_batch_size": 1,
    "project_name": project_name,
}

metric_def = [
    {"Name": "train:loss", "Regex": "train_loss=(.*?)[,\]]"},
    {"Name": "val:loss", "Regex": "val_loss=(.*?)[,\]]"},
    {"Name": "learning_rate", "Regex": "lr=(.*?)[,\]]"},
    {"Name": "val:acc", "Regex": "val_acc=(.*?)[,\]]"},
    {"Name": "val:jaccard", "Regex": "val_jaccard=(.*?)[,\]]"},
    {"Name": "val:dice", "Regex": "val_dice=(.*?)[,\]]"},
    {"Name": "test:loss", "Regex": "test_loss=(.*?)[,\]]"},
    {"Name": "test:acc", "Regex": "test_acc=(.*?)[,\]]"},
    {"Name": "test:jaccard", "Regex": "test_jaccard=(.*?)[,\]]"},
    {"Name": "test:dice", "Regex": "test_dice=(.*?)[,\]]"},
]


estimator = PyTorch(
    entry_point="train.py",
    # Point to the repo root one level up,
    # this notebook is at {repo}/notebooks/LaunchTrainingJob.ipynb
    source_dir="../",  
    role=role,
    framework_version="2.0.1",
    py_version="py310",
    instance_count=1,
    instance_type= 'ml.g4dn.4xlarge', #'ml.p3.8xlarge', #'ml.p3.16xlarge', #'ml.p4de.24xlarge', # 80GB A100 GPU   "ml.g4dn.4xlarge", 
    hyperparameters=hyperparameters, 
    # Default input_mode is "File" mode which will download a copy of all contents in S3
    # locally prior to training. "FastFile" will instead stream files from S3 as they are
    # requested. This essentially makes S3 a mounted drive to stream the data from.
    # More information can be found here: 
    # https://docs.aws.amazon.com/sagemaker/latest/dg/model-access-training-data.html
    #input_mode="File",
    metric_definitions=metric_def,
    environment={"WANDB_API_KEY": "33ae28acc6b5063bcc215f3928bcf533fd87107d"},
    #tensorboard_output_config=tensorboard_output_config,
)

estimator.fit(
    {"train_image": train_image, "train_mask": train_mask}
)

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\zelinski1\AppData\Local\sagemaker\sagemaker\config.yaml


INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: pytorch-training-2024-01-15-23-01-56-163


2024-01-15 23:02:00 Starting - Starting the training job...
2024-01-15 23:02:14 Starting - Preparing the instances for training......
2024-01-15 23:03:16 Downloading - Downloading input data......
2024-01-15 23:04:26 Downloading - Downloading the training image.........
2024-01-15 23:05:45 Training - Training image download completed. Training in progress...bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2024-01-15 23:06:25,843 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2024-01-15 23:06:25,864 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2024-01-15 23:06:25,874 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2024-01-15 23:06:25,875 sagemaker_pytorch_container.training INFO     Invoking user training script.
2024-01-15 23:06:27,343 sagemaker-training-toolkit INFO     Installing dependencies fr

EndpointConnectionError: Could not connect to the endpoint URL: "https://api.sagemaker.us-west-2.amazonaws.com/"